# 01 Data preparation

This notebook downloads daily prices from the Polygon API for every stock that was in the S&P 500 at some point between 2004 and 2024, then cuts each stock's history down to the days it was in the index. I keep only open and close prices, because the analysis works from daily returns.

Output: one CSV per ticker in `data/interim/filtered_sp500_membership/`.

The pre-2019 part of this pull came back incomplete. Polygon returned about 40% of trading days and still reported success, and nothing in this notebook checks for that. `src/repull_polygon.py` re-pulls the window that is still available and asserts coverage, and `src/merge_repull.py` joins the two pulls. A current free-tier key can no longer reproduce the pre-2019 data, so the cells below have no saved outputs.

## Imports and prerequisites

In [ ]:
import pandas as pd
import polygon 
from dotenv import load_dotenv

import os
from pathlib import Path

In [ ]:
#Prerequisites
load_dotenv("../.env")
POLYGON_KEY = os.getenv("POLYGON_KEY")

## Processing the constituents

In [ ]:
smp500_constituents= pd.read_csv("../data/raw/sp500_historical_constituents.csv")

In [ ]:
smp500_constituents.head()

Each row of the constituents file is a snapshot taken on a date when the index changed, so membership on the days in between has to be filled in later (see the membership filter at the end of the notebook).

In [ ]:
# Remove the rows which are outside our desired date range (2004-01-01 to 2024-12-31) 
smp500_constituents = smp500_constituents[(smp500_constituents['date'] >= '2004-01-01') & (smp500_constituents['date'] <= '2024-12-31')]

In [ ]:
smp500_constituents.head()

In [ ]:
#Check for missing values
smp500_constituents.isnull().sum()

The columns are strings (dates and comma-separated ticker lists), so `describe()` has nothing useful to report.

In [ ]:
smp500_constituents["tickers"] = smp500_constituents["tickers"].str.split(",")

In [ ]:
# Every ticker that appears in any snapshot
unique_tickers = smp500_constituents['tickers'].explode().unique()
unique_tickers

## Data collection

This section downloads daily bars for each ticker. Yahoo Finance would also work, but I already had a Polygon key.

In [ ]:
DATA_DIR = Path("../data/raw/polygon_initial")
DATA_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
client = polygon.StocksClient(POLYGON_KEY, True)

In [ ]:
async def fetch_ticker(ticker):
    try:
        bars = await client.get_aggregate_bars(
            ticker,
            '2004-01-01',
            '2024-12-31',
            full_range=True
        )

        df = pd.DataFrame(bars)

        df.to_csv(DATA_DIR / f"{ticker}.csv", index=False)

    except Exception as e:
        print(f"{ticker}: {e}")

for ticker in unique_tickers:
    await fetch_ticker(ticker)

## Reformatting and filtering by date

In [ ]:
# Rewrite the files into the structure of date,open,close and filter
STOCK_DATA_REFORMATTED = Path("../data/interim/reformatted_date_open_close")
STOCK_DATA_REFORMATTED.mkdir(parents=True, exist_ok=True)
for file in DATA_DIR.glob("*.csv"):
    try: # in case a file is malformed or missing data
        df = pd.read_csv(file)
        df = df[[ 'o', 'c' , "date"]] #Extracting only the columns we need
        df = df.iloc[:, [2, 0, 1]]
        df.columns = ['date', 'open', 'close']
        df = df[['date', 'open', 'close']]
        #Save the file into stock_data_reformatted
        df.to_csv(STOCK_DATA_REFORMATTED / f"{file.stem}.csv", index=False)
    except Exception as e:
        print(f"Error reading {file}: {e}")
        #Delete said file
        os.remove(file)
        continue

The raw files start on 2003-12-09, so I drop everything before 2004-01-01.

In [ ]:
#Filter the files in stock_data_filtered_by_date to only include data from 2004-01-01 to 2024-12-31
INPUT_DIR = Path("../data/interim/reformatted_date_open_close")
STOCK_DATA_FILTERED_BY_DATE = Path("../data/interim/filtered_by_date")
STOCK_DATA_FILTERED_BY_DATE.mkdir(parents=True, exist_ok=True)
for file in INPUT_DIR.glob("*.csv"):
    try:
        df = pd.read_csv(file)
        df = df[(df['date'] >= '2004-01-01') & (df['date'] <= '2024-12-31')]
        df.to_csv(STOCK_DATA_FILTERED_BY_DATE / f"{file.stem}.csv", index=False)
    except Exception as e:
        print(f"Error reading {file}: {e}")
        #Delete said file
        os.remove(file)
        continue

After filtering, the first date in the data is 2004-01-07.

## Filtering to index membership

Each stock keeps only the days on which it was in the S&P 500. Dropping stocks that left the index would bring in survivorship bias, so companies that were removed or failed, such as Bear Stearns and Countrywide, stay in the data for the years they were members.

In [ ]:

# Rereading the data to ensure that the data is in the correct format and that there are no missing values
historical_constituents = pd.read_csv(
    "../data/raw/sp500_historical_constituents.csv",
    parse_dates=["date"]
).sort_values("date")

historical_constituents["tickers"] = historical_constituents["tickers"].str.split(",")


long = (
    historical_constituents
    .explode("tickers")
)

long["tickers"] = long["tickers"].str.strip()

In [ ]:
snapshot_dates = historical_constituents["date"].sort_values().unique()

membership = {}

for ticker, grp in long.groupby("tickers"):
    dates = sorted(grp["date"])

    intervals = []

    for d in dates:
        pos = snapshot_dates.searchsorted(d)

        if pos + 1 < len(snapshot_dates):
            end = snapshot_dates[pos + 1]
        else:
            end = pd.Timestamp.max

        intervals.append((d, end))

    membership[ticker] = intervals

In [ ]:
OUT_DIR= Path("../data/interim/filtered_sp500_membership")
OUT_DIR.mkdir(parents=True, exist_ok=True)

for file in STOCK_DATA_FILTERED_BY_DATE.glob("*.csv"):

    ticker = file.stem.upper()

    if ticker not in membership:
        continue

    df = pd.read_csv(file, parse_dates=["date"])

    keep = pd.Series(False, index=df.index)

    for start, end in membership[ticker]:
        keep |= (df["date"] >= start) & (df["date"] < end)

    filtered = df[keep]

    filtered.to_csv(OUT_DIR / file.name, index=False)

## Next steps

The membership-filtered files go through `src/repull_polygon.py`, `src/merge_repull.py` and `src/build_db.py` before notebook 02 uses them. See the Method section of `docs/analysis.md` for the full order.